In [ ]:
import torch

device = torch.device("cpu")

# Check for accelerator availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA for training.")
else:
    print("Using CPU for training.")

In [ ]:
import gradio as gr
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks
from pathlib import Path

from dataset_transforms import get_transforms
from model import get_model_instance_segmentation

import pandas as pd


num_classes = 2
model_path = "faster-rcnn-20240427_2343.pth"

model = get_model_instance_segmentation(num_classes).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))

eval_transform = get_transforms(is_train=False)


def threshold_predictions(pred, confidence_threshold):
    # Create a mask for scores above the threshold
    score_mask = pred["scores"] > confidence_threshold

    # Filter the labels and boxes using the mask
    filtered_labels = [label for label, keep in zip(pred["labels"], score_mask) if keep]
    filtered_boxes = pred["boxes"][score_mask]

    new_preds = {
        "labels": filtered_labels,
        "scores": pred["scores"][score_mask],
        "boxes": filtered_boxes,
    }

    if "masks" in pred:
        new_preds["masks"] = pred["masks"][score_mask]

    return new_preds


def predict(image, confidence_threshold):
    # convert numpy image to tensor
    image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

    model.eval()
    with torch.no_grad():
        x = eval_transform(image)
        # convert RGBA -> RGB and move to device
        x = x[:3, ...].to(device)
        predictions = model([x])
        pred = predictions[0]

    image = (255.0 * (image - image.min()) / (image.max() - image.min())).to(
        torch.uint8
    )
    image = image[:3, ...]

    pred = threshold_predictions(pred, confidence_threshold)

    box_labels = [
        f"{label}:{score:.3f}" for label, score in zip(pred["labels"], pred["scores"])
    ]

    boxes = pred["boxes"].long()
    masks = (pred["masks"] > 0.7).squeeze(1) if "masks" in pred else None

    if masks is not None:
        image = draw_segmentation_masks(image, masks, alpha=0.5, colors="yellow")

    image = draw_bounding_boxes(
        image,
        boxes,
        box_labels,
        colors="#ee0044",
        width=4,
        font_size=52,
        font=str(Path("../assets/zed-mono-bold.ttf")),
    )

    np_image = image.permute(1, 2, 0).numpy()
    total_colonies = sum([int(label) for label in pred["labels"]])

    count_dict = {}

    label_list = [int(label) for label in pred["labels"]]

    for label in label_list:
        if label in count_dict:
            count_dict[label] += 1
        else:
            count_dict[label] = 1

    # Create a dataframe with x as the label and y as how many times it appears
    df = pd.DataFrame(
        {"Group size": count_dict.keys(), "Occurrences": count_dict.values()}
    )

    return np_image, total_colonies, df


demo = gr.Interface(
    fn=predict,
    inputs=[
        "image",
        gr.Slider(label="Confidence threshold", maximum=1, value=0.9, step=0.05),
    ],
    examples=[
        ["../data/unlabeled/default_mmrsbpb_default_19012024135839.jpg"],
        ["../data/unlabeled/default_mmrsbpb_default_19012024140321.jpg"],
        ["../data/unlabeled/default_mmrsbpb_default_19012024140822.jpg"],
        ["../data/unlabeled/default_mmrsbpb_mMRS-BPB_06122023155324.jpg"],
    ],
    outputs=[
        gr.Image(label="Segmented image"),
        gr.Number(label="Number of colonies detected"),
        gr.BarPlot(label="Group size distribution", x="Group size", y="Occurrences"),
    ],
    title="Colony Detection",
    allow_flagging="never",
)

demo.launch(inline=False, server_name="0.0.0.0")